# Train PatchTST + CVAE on Colab

Run this notebook's kernel connected to a Colab runtime (VS Code: kernel picker top-right -> "Select Another Kernel" -> Google Colab -> pick a GPU runtime).

Run the cells top to bottom. The clone step is idempotent (pulls if already cloned).

In [34]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB


In [35]:
import os

REPO_URL = "https://github.com/WoodyChang21/ECE1508_GenAI.git"
BRANCH = "steven2"

if not os.path.isdir("ECE1508_GenAI"):
    !git clone -b {BRANCH} {REPO_URL}
else:
    # Not a plain `git pull` -- if this clone has ANY local changes (e.g. leftover
    # checkpoints/outputs from an earlier run in the same runtime that never got pushed),
    # a pull can fail outright ("local changes would be overwritten") and Colab just
    # prints the error and moves on -- training then silently proceeds on stale code with
    # no visible failure until much later (e.g. a non-fast-forward push at the end).
    # fetch + hard reset guarantees this checkout exactly matches origin/{BRANCH} no
    # matter what state it was left in.
    !cd ECE1508_GenAI && git fetch origin {BRANCH} && git reset --hard origin/{BRANCH}

%cd ECE1508_GenAI
!git log --oneline -1


Cloning into 'ECE1508_GenAI'...
remote: Enumerating objects: 1493, done.
remote: Counting objects: 100% (1005/1005), done.
remote: Compressing objects: 100% (724/724), done.
remote: Total 1493 (delta 514), reused 754 (delta 280), pack-reused 488 (from 1)
Receiving objects: 100% (1493/1493), 53.58 MiB | 33.64 MiB/s, done.
Resolving deltas: 100% (715/715), done.
/content/ECE1508_GenAI/ECE1508_GenAI/ECE1508_GenAI/ECE1508_GenAI
5abf8d3 (HEAD -> steven2, origin/steven2) Add auxiliary direction loss for CVAE; record decoder_ctx_dim didn't fix collapse


In [36]:
# torch is preinstalled on Colab; just need mplfinance + pyyaml
!pip install -q mplfinance pyyaml

In [37]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


## Sanity checks (data pipeline tests)

Cheap to run first -- confirms the feature/window logic before committing to a long training run.

In [25]:
!pip install -q pytest
!python -m pytest steven/tests/ -v

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/ECE1508_GenAI/ECE1508_GenAI/ECE1508_GenAI
plugins: anyio-4.14.2, typeguard-4.5.2, langsmith-0.10.2
collected 39 items                                                             

steven/tests/test_cvae_inpainting.py::test_default_decoder_ctx_dim_matches_old_behavior PASSED [  2%]
steven/tests/test_cvae_inpainting.py::test_decoder_ctx_dim_bottlenecks_decoder_input PASSED [  5%]
steven/tests/test_cvae_inpainting.py::test_prior_head_still_sees_full_ctx_dim_when_bottlenecked PASSED [  7%]
steven/tests/test_data_pipeline.py::test_reconstruct_prices_round_trip PASSED [ 10%]
steven/tests/test_data_pipeline.py::test_anchor_correction_matches_close_0_for_all_horizon_bars PASSED [ 12%]
steven/tests/test_data_pipeline.py::test_wick_components_non_negative PASSED [ 15%]
steven/tests/test_data_pip

## Train PatchTST (real defaults: 20k windows/epoch, 20 epochs, configs/patchtst.yaml)

Drop `--max-epochs`/`--train-windows-per-epoch` overrides below if you want a quick smoke run first instead of the full config.

In [26]:
!python steven/src/train_patchtst.py --config steven/configs/patchtst.yaml --device auto

05:44:51 device: cuda
05:44:52 Gap report: 145 fully missing weekdays, 39 short sessions (<7 bars)
05:44:52 Dropping first row (2010-01-04 09:30:00): no previous close to compute a return from
05:44:55 epoch 1/20  train_loss=0.23157  val_loss=0.10953  (2.4s)
05:44:55   -> saved best checkpoint (val_loss=0.10953) to steven/outputs/patchtst_checkpoint.pt
05:44:57 epoch 2/20  train_loss=0.16731  val_loss=0.11391  (1.9s)
05:44:59 epoch 3/20  train_loss=0.15137  val_loss=0.09968  (1.9s)
05:44:59   -> saved best checkpoint (val_loss=0.09968) to steven/outputs/patchtst_checkpoint.pt
05:45:01 epoch 4/20  train_loss=0.14287  val_loss=0.09491  (2.0s)
05:45:01   -> saved best checkpoint (val_loss=0.09491) to steven/outputs/patchtst_checkpoint.pt
05:45:03 epoch 5/20  train_loss=0.14188  val_loss=0.09595  (1.9s)
05:45:05 epoch 6/20  train_loss=0.13820  val_loss=0.09251  (1.9s)
05:45:05   -> saved best checkpoint (val_loss=0.09251) to steven/outputs/patchtst_checkpoint.pt
05:45:07 epoch 7/20  train_

## Train CVAE (real defaults: 20k windows/epoch, 30 epochs, configs/cvae.yaml)

In [38]:
!python steven/src/train_cvae.py --config steven/configs/cvae.yaml --device auto

05:58:46 device: cuda
05:58:47 Gap report: 145 fully missing weekdays, 39 short sessions (<7 bars)
05:58:47 Dropping first row (2010-01-04 09:30:00): no previous close to compute a return from
05:58:47 price_scale (open_ret, body_ret, upper_wick, lower_wick): [0.002682909369468689, 0.0028937843162566423, 0.0016679799882695079, 0.0017456382047384977]
05:58:50 epoch 1/30  beta=0.20  train_recon=364.28162 (kl=23.4602 dir=2.0484)  val_recon=10.67368 (kl=10.9213 dir=1.4672)  (2.0s)
05:58:50   -> saved best checkpoint (val_recon=10.67368) to steven/outputs/cvae_checkpoint.pt
05:58:51 epoch 2/30  beta=0.40  train_recon=5.43239 (kl=5.8916 dir=1.1354)  val_recon=3.65514 (kl=3.3669 dir=0.8597)  (1.4s)
05:58:51   -> saved best checkpoint (val_recon=3.65514) to steven/outputs/cvae_checkpoint.pt
05:58:53 epoch 3/30  beta=0.60  train_recon=3.86754 (kl=2.2307 dir=0.8848)  val_recon=3.51302 (kl=1.4648 dir=0.8048)  (1.4s)
05:58:53   -> saved best checkpoint (val_recon=3.51302) to steven/outputs/cvae_ch

## Daily-bars CVAE probe (context-length sweep, 2023-2025 test period)

Tests whether CVAE's direction collapse is a modeling problem or a sampling-frequency
problem: same model recipe as `configs/cvae.yaml` above, but on SPY resampled to daily bars
(from the same hourly parquet, no new data collection), sweeping context length across
5/10/15/20 trading days, test period extended to 2023-01-01 -- 2025-05-30 (one more year
than the hourly project). CVAE only -- PatchTST's architecture is hardcoded to the hourly
context length (see `probe_daily_cvae.py`'s module docstring), out of scope here. Does not
touch or overwrite the hourly `cvae_checkpoint.pt` above (writes to
`cvae_checkpoint_daily.pt` instead). See `daily_signal_probe.md` for the full analysis and
`probe_daily_cvae.py` for what this actually runs.

In [ ]:
import sys
sys.path.insert(0, "steven")
import probe_daily_cvae as pdc
from IPython.display import Markdown, display

results = pdc.main()
display(Markdown(pdc.format_report(results)))

## Evaluate both models on the fixed test set

In [39]:
!python steven/src/evaluate.py \
  --patchtst-checkpoint steven/outputs/patchtst_checkpoint.pt \
  --cvae-checkpoint steven/outputs/cvae_checkpoint.pt \
  --device auto

05:59:49 device: cuda
05:59:49 Gap report: 145 fully missing weekdays, 39 short sessions (<7 bars)
05:59:49 Dropping first row (2010-01-04 09:30:00): no previous close to compute a return from
05:59:49 sell-price shrink bound: p99.0 of |anchored log return| over train = 0.0190 (vs. model's own MAX_LOG_RETURN)
05:59:49 running walk-forward backtest (ctx=70 bars, patchtst_min_return>=0.100%, cvae_min_return>=0.020% [no CVAE quality gate -- see backlog.md], stop_loss=2.00% [shared, see backlog.md], 24537..27006)...
06:00:01 walk-forward: PatchTST 460 trades / 1477 decisions, CVAE 0 trades / 2397 decisions
06:00:01 wrote metrics to steven/outputs/metrics.json
06:00:01 walk_forward: {
  "ctx_bars": 70,
  "patchtst_min_return_threshold": 0.001,
  "cvae_min_return_threshold": 0.0002,
  "stop_loss_pct": 0.02,
  "buy_and_hold": {
    "entry_date": "2024-01-16",
    "entry_price": 474.95,
    "exit_date": "2025-05-30",
    "exit_price": 589.46,
    "elapsed_years": 1.3689253935660506,
    "total

## Refresh v1.md from this run

Rewrites the Results/backtest tables and sample images in `steven/v1.md` from the metrics.json + sample_plots this run just produced (see `steven/src/update_report.py`). Only the tables/images are rewritten -- surrounding prose (interpretation, caveats) is left as-is; review it by hand if the story changed. This only edits the file in the cloned repo here -- push/download separately if you want to keep it.

In [40]:
!python steven/src/update_report.py

06:04:23 updated steven/v1.md: results-samples, hit-summary, spread-summary, buy-hold-benchmark, walk-forward-strategies, walk-forward-outcome-breakdown
06:04:23 not auto-updated -- reread and edit by hand if the story changed: the 'In plain terms' / 'A subtle but important point' interpretation paragraphs under Results, the 'pre-retrain checkpoints' caveat in Results, and the 'Retrain both models' checkbox under Next steps.


## Sync results back to GitHub

Commits `steven/outputs/` (checkpoints, metrics.json, sample_plots) and the regenerated `steven/v1.md` from this Colab runtime and pushes straight to the `steven2` branch -- no manual zip/download step. That step wasn't reliably reaching the local machine: `files.download()`'s browser-download trick only works from the Colab web UI, not when this kernel is attached remotely (e.g. from VS Code's kernel picker), so nothing ever landed on disk.

Needs a GitHub personal access token with `repo` write scope for this push only -- entered via `getpass` below, never written to the notebook or committed anywhere.

In [41]:
# %%bash
# git fetch origin steven2
# git merge origin/steven2 --no-edit

In [42]:
import getpass

token = getpass.getpass("GitHub PAT (repo write, used only for this push): ")

In [43]:
%%bash -s "$token"
TOKEN="$1"
if [ -z "$TOKEN" ]; then
  echo "Token was empty -- re-run the getpass cell above and actually paste your PAT before pressing Enter." >&2
  exit 1
fi
git config user.email "colab@ephemeral.local"
git config user.name "Colab Runtime"
git add steven/outputs steven/v1.md
if git diff --cached --quiet; then
  echo "Nothing new to commit -- outputs/v1.md unchanged from last commit."
else
  git commit -m "Retrain + refresh results from Colab run"
fi
# Push unconditionally -- a prior run may have committed but failed to push (e.g. a blank
# token), in which case there's nothing new to commit here but HEAD is still ahead of origin.
# Pushes to steven2, not steven -- this notebook and this branch are the working copy for
# now; steven is left alone so a second person's in-flight work there can't collide with
# this runtime's pushes.
git push "https://${TOKEN}@github.com/WoodyChang21/ECE1508_GenAI.git" HEAD:steven2

[steven2 ceb78d7] Retrain + refresh results from Colab run
 5 files changed, 94 insertions(+), 94 deletions(-)
 rewrite steven/outputs/cvae_checkpoint.pt (92%)
 delete mode 100644 steven/outputs/sample_plots/no_trade_start26500_ctx70.png
 create mode 100644 steven/outputs/sample_plots/no_trade_start26557_ctx70.png
 rewrite steven/outputs/sample_plots/samples.json (64%)


To https://github.com/WoodyChang21/ECE1508_GenAI.git
   5abf8d3..ceb78d7  HEAD -> steven2


### Fallback: zip + browser download

Only useful if you're running this notebook inside the actual Colab web UI (not a remote kernel) and would rather download a zip than push through git.

In [33]:
!zip -r outputs.zip steven/outputs

try:
    from google.colab import files
    files.download("outputs.zip")
except ImportError:
    print("Not in a Colab frontend session -- outputs.zip is in the working dir, grab it manually.")

  adding: steven/outputs/ (stored 0%)
  adding: steven/outputs/metrics.json (deflated 68%)
  adding: steven/outputs/cvae_checkpoint.pt (deflated 8%)
  adding: steven/outputs/sample_plots/ (stored 0%)
  adding: steven/outputs/sample_plots/samples.json (deflated 69%)
  adding: steven/outputs/sample_plots/no_trade_start26500_ctx70.png (deflated 10%)
  adding: steven/outputs/patchtst_checkpoint.pt (deflated 9%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Optional: reproduce the pre-fix CVAE checkpoint (comparison experiment)

Commit `2c4ad99` (before `price_scale`, `decoder_ctx_dim`, or `w_direction` existed) is the
best walk-forward result CVAE has produced (+6.79% total return, 66.4% trade rate). Every fix
attempted since has made the loss more "correct" by the direction-collapse diagnosis but
hasn't matched it. `configs/cvae_pre_fix_repro.yaml` reproduces that exact config (writes to
a separate checkpoint so it doesn't clobber `configs/cvae.yaml`'s), and
`src/diagnose_cvae_direction.py` reruns the same variance/correlation/eligibility checks used
throughout `cvae_direction_collapse.md` against any checkpoint -- run it against both to see
whether the fixes actually helped or whether `2c4ad99` was a favorable roll against one test
path. See `cvae_direction_collapse.md`'s "Revisiting the pre-collapse-chasing checkpoint".

In [ ]:
!python steven/src/train_cvae.py --config steven/configs/cvae_pre_fix_repro.yaml --device auto

In [ ]:
print("=== current checkpoint (price_scale + decoder_ctx_dim + w_direction) ===")
!python steven/src/diagnose_cvae_direction.py --cvae-checkpoint steven/outputs/cvae_checkpoint.pt

print("\n=== pre-fix repro checkpoint (matches commit 2c4ad99) ===")
!python steven/src/diagnose_cvae_direction.py --cvae-checkpoint steven/outputs/cvae_checkpoint_pre_fix_repro.pt